# Perbandingan Performa: Neural Network vs. Random Forest

Notebook ini bertujuan untuk membandingkan performa dua model machine learning, yaitu **Neural Network (MLP)** dan **Random Forest**, pada dataset *Online Shoppers Intention*.

**Metrik Evaluasi:**
1.  **Akurasi:** Seberapa sering model membuat prediksi yang benar.
2.  **Waktu Training:** Waktu yang dibutuhkan untuk melatih model.
3.  **Waktu Prediksi:** Waktu yang dibutuhkan model untuk membuat prediksi pada data baru.

### 1. Import Library

Langkah pertama adalah mengimpor semua library yang dibutuhkan.

In [2]:
import time
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Mengatur agar output tidak terpotong
pd.set_option('display.max_columns', None)
print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


### 2. Load & Preprocessing Data

- Dataset: `online_shoppers_intention.csv`
- Target: `Revenue`
- Preprocessing: One-Hot Encoding untuk fitur kategorikal.

In [3]:
# Load dataset
df = pd.read_csv('online_shoppers_intention.csv')

# Pisahkan fitur (X) dan target (y)
X = df.drop('Revenue', axis=1)
y = df['Revenue']

# Identifikasi fitur kategorikal dan numerik
categorical_features = X.select_dtypes(include=['object', 'bool']).columns
numerical_features = X.select_dtypes(include=np.number).columns

# One-Hot Encoding untuk fitur kategorikal
# handle_unknown='ignore' untuk mengatasi nilai yang mungkin ada di test set tapi tidak di train set
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = pd.DataFrame(encoder.fit_transform(X[categorical_features]))
X_encoded.columns = encoder.get_feature_names_out(categorical_features)

# Gabungkan kembali fitur numerik dan yang sudah di-encode
X = X.drop(categorical_features, axis=1)
X = pd.concat([X, X_encoded], axis=1)

# Ubah nama kolom boolean hasil encoding agar lebih informatif
X.rename(columns={'Month_Aug': 'Month_Aug', 'Month_Dec': 'Month_Dec', 'Month_Feb': 'Month_Feb', 
                   'Month_Jul': 'Month_Jul', 'Month_June': 'Month_June', 'Month_Mar': 'Month_Mar', 
                   'Month_May': 'Month_May', 'Month_Nov': 'Month_Nov', 'Month_Oct': 'Month_Oct', 
                   'Month_Sep': 'Month_Sep', 'VisitorType_New_Visitor': 'VisitorType_New_Visitor', 
                   'VisitorType_Other': 'VisitorType_Other', 'VisitorType_Returning_Visitor': 'VisitorType_Returning_Visitor',
                   'Weekend_False': 'Weekend_False', 'Weekend_True': 'Weekend_True'}, inplace=True)

print("Dimensi data setelah preprocessing:", X.shape)
X.head()

Dimensi data setelah preprocessing: (12330, 29)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType,Month_Aug,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,VisitorType_New_Visitor,VisitorType_Other,VisitorType_Returning_Visitor,Weekend_False,Weekend_True
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,1,1,1,1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,2,2,1,2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,4,1,9,3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,3,2,2,4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,3,3,1,4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


### 3. Split Data (Train-Test)

- Pembagian data: 80% training, 20% testing.
- `stratify=y`: Memastikan proporsi kelas pada data training dan testing sama dengan proporsi pada data asli. Penting untuk data yang tidak seimbang.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Distribusi kelas pada y_train:\n", y_train.value_counts(normalize=True))
print("\nDistribusi kelas pada y_test:\n", y_test.value_counts(normalize=True))

Distribusi kelas pada y_train:
 Revenue
False    0.845296
True     0.154704
Name: proportion, dtype: float64

Distribusi kelas pada y_test:
 Revenue
False    0.845093
True     0.154907
Name: proportion, dtype: float64


### 4. Penanganan Data Imbalance (SMOTE)

- **SMOTE (Synthetic Minority Over-sampling Technique):** Digunakan untuk mengatasi ketidakseimbangan kelas.
- **Penting:** SMOTE hanya diterapkan pada data training untuk mencegah *data leakage* (informasi dari data testing bocor ke proses training).

In [5]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Distribusi kelas setelah SMOTE:\n", y_train_res.value_counts(normalize=True))

Distribusi kelas setelah SMOTE:
 Revenue
False    0.5
True     0.5
Name: proportion, dtype: float64


### 5. Scaling Data (StandardScaler)

- **StandardScaler:** Mengubah skala fitur sehingga memiliki rata-rata 0 dan standar deviasi 1.
- **Penting:** Scaling hanya diperlukan untuk model Neural Network.
- Scaler di-`fit` hanya pada data training (`X_train_res`) dan kemudian digunakan untuk mentransformasi `X_train_res` dan `X_test`.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

### 6. Model 1: Neural Network (TensorFlow MLP)

- **Arsitektur:**
    - Input Layer
    - Dense(128, ReLU) + Dropout(0.5)
    - Dense(64, ReLU) + Dropout(0.3)
    - Output Layer: Dense(1, Sigmoid) -> untuk klasifikasi biner.
- **Kompilasi:**
    - `optimizer='adam'`
    - `loss='binary_crossentropy'`
    - `metrics=['accuracy']`
- **EarlyStopping:** Menghentikan training jika tidak ada peningkatan pada `val_loss` setelah 5 epoch (`patience=5`) untuk mencegah overfitting.

In [7]:
# Mendefinisikan model
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# Kompilasi model
nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Callback untuk Early Stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# --- Training ---
start_time_nn_train = time.time()
history = nn_model.fit(
    X_train_scaled, y_train_res,
    epochs=50,
    batch_size=32,
    validation_split=0.2, # Menggunakan sebagian data training untuk validasi
    callbacks=[early_stopping],
    verbose=1
)
end_time_nn_train = time.time()
runtime_nn_train = end_time_nn_train - start_time_nn_train

# --- Prediksi ---
start_time_nn_pred = time.time()
y_pred_nn_proba = nn_model.predict(X_test_scaled)
y_pred_nn = (y_pred_nn_proba > 0.5).astype(int) # Konversi probabilitas ke kelas biner
end_time_nn_pred = time.time()
runtime_nn_pred = end_time_nn_pred - start_time_nn_pred

# --- Evaluasi ---
accuracy_nn = accuracy_score(y_test, y_pred_nn)

print(f"\nNeural Network Training Time: {runtime_nn_train:.4f} seconds")
print(f"Neural Network Prediction Time: {runtime_nn_pred:.4f} seconds")
print(f"Neural Network Accuracy: {accuracy_nn:.4f}")

Epoch 1/50


/Users/mac/sc_env/sc2_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


417/417 ━━━━━━━━━━━━━━━━━━━━ 1s 797us/step - accuracy: 0.7723 - loss: 0.4643 - val_accuracy: 0.7566 - val_loss: 0.4883
Epoch 2/50
417/417 ━━━━━━━━━━━━━━━━━━━━ 0s 658us/step - accuracy: 0.8313 - loss: 0.3807 - val_accuracy: 0.7884 - val_loss: 0.4656
Epoch 3/50
417/417 ━━━━━━━━━━━━━━━━━━━━ 0s 632us/step - accuracy: 0.8451 - loss: 0.3556 - val_accuracy: 0.8237 - val_loss: 0.4203
Epoch 4/50
417/417 ━━━━━━━━━━━━━━━━━━━━ 0s 648us/step - accuracy: 0.8587 - loss: 0.3342 - val_accuracy: 0.8201 - val_loss: 0.4183
Epoch 5/50
417/417 ━━━━━━━━━━━━━━━━━━━━ 0s 582us/step - accuracy: 0.8654 - loss: 0.3216 - val_accuracy: 0.8492 - val_loss: 0.3606
Epoch 6/50
417/417 ━━━━━━━━━━━━━━━━━━━━ 0s 571us/step - accuracy: 0.8733 - loss: 0.3077 - val_accuracy: 0.8639 - val_loss: 0.3382
Epoch 7/50
417/417 ━━━━━━━━━━━━━━━━━━━━ 0s 571us/step - accuracy: 0.8731 - loss: 0.3020 - val_accuracy: 0.8852 - val_loss: 0.2973
Epoch 8/50
417/417 ━━━━━━━━━━━━━━━━━━━━ 0s 640us/step - accuracy: 0.8777 - loss: 0.2926 - val_accurac

### 7. Model 2: Random Forest

- **Parameter:**
    - `n_estimators=300`: Jumlah pohon dalam forest.
    - `max_depth=15`: Kedalaman maksimum setiap pohon.
    - `random_state=42`: Untuk reproduktifitas hasil.
- **Data:** Menggunakan data yang sudah di-SMOTE tetapi **tidak di-scaling** (`X_train_res`, `y_train_res`). Random Forest tidak sensitif terhadap skala fitur.

In [8]:
# Mendefinisikan model
rf_model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)

# --- Training ---
start_time_rf_train = time.time()
rf_model.fit(X_train_res, y_train_res)
end_time_rf_train = time.time()
runtime_rf_train = end_time_rf_train - start_time_rf_train

# --- Prediksi ---
start_time_rf_pred = time.time()
y_pred_rf = rf_model.predict(X_test)
end_time_rf_pred = time.time()
runtime_rf_pred = end_time_rf_pred - start_time_rf_pred

# --- Evaluasi ---
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest Training Time: {runtime_rf_train:.4f} seconds")
print(f"Random Forest Prediction Time: {runtime_rf_pred:.4f} seconds")
print(f"Random Forest Accuracy: {accuracy_rf:.4f}")

Random Forest Training Time: 0.8411 seconds
Random Forest Prediction Time: 0.0394 seconds
Random Forest Accuracy: 0.8970


### 8. Perbandingan Hasil

Menampilkan hasil perbandingan kedua model dalam format tabel yang rapi.

In [9]:
# Membuat DataFrame untuk perbandingan
results = {
    'Model': ['Neural Network (MLP)', 'Random Forest'],
    'Accuracy': [accuracy_nn, accuracy_rf],
    'Runtime Training (s)': [runtime_nn_train, runtime_rf_train],
    'Runtime Prediction (s)': [runtime_nn_pred, runtime_rf_pred]
}

comparison_df = pd.DataFrame(results)

# Menampilkan tabel perbandingan
comparison_df.set_index('Model', inplace=True)
comparison_df.style.format({
    'Accuracy': '{:.4f}',
    'Runtime Training (s)': '{:.4f}',
    'Runtime Prediction (s)': '{:.4f}'
})

,Accuracy,Runtime Training (s),Runtime Prediction (s)
Model,,,
Neural Network (MLP),0.8824,8.1125,0.1174
Random Forest,0.8970,0.8411,0.0394
